<a href="https://colab.research.google.com/github/Leashaniya/Research-Project/blob/leasha/4_LECTURE_SLIDES%E2%86%92_CHUNKS_%2B_EMBEDDINGS_%2B_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================================
# NOTEBOOK 4 — LECTURE SLIDES (GREEN-BOX FIGURES) → CHUNKS + EMBEDDINGS + FAISS
#
# Input:
#   /content/drive/MyDrive/lectureslides/*.pdf
#
# Output (all inside RP folder):
#   /content/drive/MyDrive/RP/lecture_slides_extraction/<pdf_stem>/
#       pages_text/slide_001_text.txt ...
#       figures/slide_001_fig_1.png ...
#       slides_text_with_figures.txt
#
#   /content/drive/MyDrive/RP/lecture_slides_extraction/3
#       slides_chunks.jsonl
#       slides_chunks_index.csv
#
#   /content/drive/MyDrive/RP/slides_embeddings/
#       slides_embeddings.npy
#       slides_metadata.jsonl
#       slides_faiss_index_flatip.index
# ==========================================================

# -------------------------------
# 0) Mount Drive
# -------------------------------
from google.colab import drive
drive.mount("/content/drive")

# -------------------------------
# 1) Install deps
# -------------------------------
!pip install --quiet pymupdf opencv-python-headless pytesseract tqdm sentence-transformers faiss-cpu

# (Optional but recommended on Colab)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-eng

# -------------------------------
# 2) Imports
# -------------------------------
import os, re, json, csv
from pathlib import Path
from tqdm import tqdm
import numpy as np
import cv2
import fitz  # PyMuPDF
import pytesseract
from pytesseract import Output
from sentence_transformers import SentenceTransformer
import faiss

# -------------------------------
# 3) Paths & Config
# -------------------------------
SLIDES_DIR = Path("/content/drive/MyDrive/lectureslides")
OUT_ROOT   = Path("/content/drive/MyDrive/RP/lecture_slides_extraction")
EMB_ROOT   = Path("/content/drive/MyDrive/RP/slides_embeddings")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
EMB_ROOT.mkdir(parents=True, exist_ok=True)

DPI = 220

# green HSV range (tunable)
# Works for most "marker green box" highlights.
GREEN_LOW  = np.array([35, 40, 40])
GREEN_HIGH = np.array([90, 255, 255])

MIN_AREA = 2500        # ignore tiny noise
PAD = 6                # expand rectangle slightly
BORDER_STRIP = 4       # shrink crop inward to remove the green border
OCR_TEXT_MIN_CHARS = 40
TESS_CONFIG = "--oem 3 --psm 6"

CHUNK_WORDS = 350
OVERLAP_WORDS = 70

# -------------------------------
# 4) Utilities
# -------------------------------
def detect_green_boxes(img_bgr, low=GREEN_LOW, high=GREEN_HIGH, min_area=MIN_AREA, pad=PAD):
    """Return list of rects in IMAGE coords: {x,y,w,h,y_mid}."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, low, high)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    rects = []
    h, w = img_bgr.shape[:2]
    for c in contours:
        x,y,ww,hh = cv2.boundingRect(c)
        if ww*hh < min_area:
            continue
        x0 = max(0, x - pad)
        y0 = max(0, y - pad)
        x1 = min(w, x + ww + pad)
        y1 = min(h, y + hh + pad)
        rects.append({
            "x": int(x0), "y": int(y0), "w": int(x1-x0), "h": int(y1-y0),
            "y_mid": int(y0 + (y1-y0)//2)
        })

    # sort top-to-bottom, left-to-right
    rects = sorted(rects, key=lambda r: (r["y"], r["x"]))
    return rects

def rect_img_to_pdf(r_img, sx, sy):
    """Map IMAGE rect -> PDF rect using scale factors (img = pdf * s)."""
    x0 = r_img["x"] / sx
    y0 = r_img["y"] / sy
    x1 = (r_img["x"] + r_img["w"]) / sx
    y1 = (r_img["y"] + r_img["h"]) / sy
    return (x0, y0, x1, y1)

def overlaps(b1, b2):
    """Axis-aligned bbox overlap (x0,y0,x1,y1)."""
    x0,y0,x1,y1 = b1
    a0,b0,a1,b1_ = b2
    return not (x1 < a0 or a1 < x0 or y1 < b0 or b1_ < y0)

def crop_strip_border(img, x, y, w, h, strip=BORDER_STRIP):
    x0 = max(0, x + strip)
    y0 = max(0, y + strip)
    x1 = min(img.shape[1], x + w - strip)
    y1 = min(img.shape[0], y + h - strip)
    if x1 <= x0 or y1 <= y0:
        return img[y:y+h, x:x+w]
    return img[y0:y1, x0:x1]

def ocr_masked_text(img_bgr, rects_img):
    """OCR fallback: mask green-rect regions then OCR remaining words (light structure by y)."""
    mask = np.ones(img_bgr.shape[:2], dtype=np.uint8) * 255
    for r in rects_img:
        cv2.rectangle(mask, (r["x"], r["y"]), (r["x"]+r["w"], r["y"]+r["h"]), 0, -1)

    img_masked = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)

    data = pytesseract.image_to_data(img_masked, config=TESS_CONFIG, output_type=Output.DICT)
    words = []
    n = len(data.get("text", []))
    for i in range(n):
        t = str(data["text"][i]).strip()
        if not t:
            continue
        try:
            y = int(data["top"][i])
        except:
            y = 0
        words.append((y, t))

    words.sort(key=lambda x: x[0])
    # simple: join as lines by y buckets
    lines = []
    cur = []
    last_y = None
    for y, t in words:
        if last_y is None:
            cur = [t]
            last_y = y
        elif abs(y - last_y) <= 10:
            cur.append(t)
            last_y = y
        else:
            lines.append(" ".join(cur))
            cur = [t]
            last_y = y
    if cur:
        lines.append(" ".join(cur))

    return "\n".join(lines).strip()

# -------------------------------
# 5) Process ONE slides PDF
# -------------------------------
def process_slides_pdf(pdf_path: Path, out_root=OUT_ROOT, dpi=DPI):
    stem = pdf_path.stem
    out_dir = out_root / stem
    pages_out = out_dir / "pages_text"
    figs_out  = out_dir / "figures"
    pages_out.mkdir(parents=True, exist_ok=True)
    figs_out.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(pdf_path))
    combined = []

    for pno in range(doc.page_count):
        page = doc[pno]

        # render to image (ensures we can detect green boxes)
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        rects_img = detect_green_boxes(img)

        # scale factors
        w_img, h_img = pix.width, pix.height
        w_pdf, h_pdf = float(page.rect.width), float(page.rect.height)
        sx = (w_img / w_pdf) if w_pdf else 1.0
        sy = (h_img / h_pdf) if h_pdf else 1.0

        # map to pdf coords for text-block filtering + placeholder positioning
        rects_pdf = []
        for ri in rects_img:
            x0,y0,x1,y1 = rect_img_to_pdf(ri, sx, sy)
            rects_pdf.append({
                "pdf_rect": (x0,y0,x1,y1),
                "y_mid_pdf": (y0+y1)/2.0,
                "img_rect": ri
            })

        # 1) Extract selectable text blocks (best path)
        blocks = page.get_text("blocks")  # x0,y0,x1,y1,text,...
        lines = []
        if blocks:
            # keep only blocks not overlapping any green rect
            for b in blocks:
                if len(b) < 5:
                    continue
                x0,y0,x1,y1,txt = b[:5]
                txt = str(txt).strip()
                if not txt:
                    continue

                bbox = (float(x0),float(y0),float(x1),float(y1))
                inside = False
                for rp in rects_pdf:
                    if overlaps(bbox, rp["pdf_rect"]):
                        inside = True
                        break
                if inside:
                    continue
                lines.append((float(y0), txt))

            lines.sort(key=lambda x: x[0])
            page_text = "\n".join([t for _,t in lines]).strip()
        else:
            page_text = ""

        # 2) If text layer is weak → OCR fallback (masked)
        if len(page_text) < OCR_TEXT_MIN_CHARS:
            page_text = ocr_masked_text(img, [r["img_rect"] for r in rects_pdf])

        # 3) Extract figure crops + insert placeholders in reading order
        placeholders = []
        for i, rp in enumerate(rects_pdf, start=1):
            ri = rp["img_rect"]
            crop = img[ri["y"]:ri["y"]+ri["h"], ri["x"]:ri["x"]+ri["w"]]
            crop = crop_strip_border(crop, 0, 0, crop.shape[1], crop.shape[0], strip=BORDER_STRIP)

            fig_name = f"slide_{pno+1:03d}_fig_{i}.png"
            fig_path = figs_out / fig_name
            cv2.imwrite(str(fig_path), crop)

            placeholders.append((rp["y_mid_pdf"], f"[FIGURE: {fig_name}]"))

        # Insert placeholders roughly by y position:
        # If we have block-based ordering, we interleave by y.
        if lines:
            merged = []
            ph_i = 0
            placeholders.sort(key=lambda x: x[0])

            for y0, txt in lines:
                while ph_i < len(placeholders) and placeholders[ph_i][0] <= y0:
                    merged.append(placeholders[ph_i][1])
                    ph_i += 1
                merged.append(txt)

            while ph_i < len(placeholders):
                merged.append(placeholders[ph_i][1])
                ph_i += 1

            final_text = "\n".join([m for m in merged if str(m).strip()]).strip()
        else:
            # OCR-only ordering (rough): put placeholders first by y, then OCR text
            final_text = page_text
            if placeholders:
                ph_text = "\n".join([p[1] for p in sorted(placeholders, key=lambda x: x[0])])
                final_text = (ph_text + "\n" + page_text).strip()

        # save per-slide
        slide_file = pages_out / f"slide_{pno+1:03d}_text.txt"
        slide_file.write_text(final_text, encoding="utf-8")

        combined.append(f"\n\n--- SLIDE {pno+1} ---\n{final_text}")

    combined_text = "".join(combined).strip()
    (out_dir / "slides_text_with_figures.txt").write_text(combined_text, encoding="utf-8")
    print(f"✅ {stem}: slides processed = {doc.page_count}, figures folder = {figs_out}")
    return True

# -------------------------------
# 6) Build SLIDES chunks.jsonl + index.csv (global)
# -------------------------------
def build_slides_chunks(out_root=OUT_ROOT, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    chunks_jsonl = out_root / "slides_chunks.jsonl"
    chunks_csv   = out_root / "slides_chunks_index.csv"

    all_text_files = sorted(out_root.glob("*/slides_text_with_figures.txt"))
    if not all_text_files:
        raise FileNotFoundError("No slides_text_with_figures.txt found. Run slide extraction first.")

    global_chunks = []
    csv_rows = []

    for tf in all_text_files:
        pdf_stem = tf.parent.name
        doc_text = tf.read_text(encoding="utf-8", errors="ignore")

        words = re.sub(r"\n", " \n ", doc_text).split()
        n = len(words)
        step = max(1, chunk_words - overlap_words)

        start = 0
        c_i = 0
        while start < n:
            end = min(start + chunk_words, n)
            chunk_words_list = words[start:end]
            chunk_text = " ".join(chunk_words_list).replace(" \n ", "\n").strip()

            # slide number heuristic: first slide marker inside chunk
            slide_matches = re.findall(r"--- SLIDE (\d+) ---", chunk_text)
            slide_no = int(slide_matches[0]) if slide_matches else None

            chunk_id = f"{pdf_stem}__sc{c_i:04d}"
            rec = {
                "chunk_id": chunk_id,
                "pdf_stem": pdf_stem,
                "slide_no": slide_no,
                "start_word": start,
                "end_word": end,
                "n_words": len(chunk_words_list),
                "text": chunk_text,
                "source": str(tf),
            }
            global_chunks.append(rec)
            csv_rows.append({
                "chunk_id": chunk_id,
                "pdf_stem": pdf_stem,
                "slide_no": slide_no,
                "start_word": start,
                "end_word": end,
                "n_words": len(chunk_words_list),
                "text_snippet": (chunk_text[:200] + "...") if len(chunk_text) > 200 else chunk_text,
            })

            c_i += 1
            start += step

    with open(chunks_jsonl, "w", encoding="utf-8") as jf:
        for rec in global_chunks:
            jf.write(json.dumps(rec, ensure_ascii=False) + "\n")

    with open(chunks_csv, "w", newline="", encoding="utf-8") as cf:
        writer = csv.DictWriter(cf, fieldnames=list(csv_rows[0].keys()) if csv_rows else ["chunk_id"])
        writer.writeheader()
        writer.writerows(csv_rows)

    print(f"✅ slides_chunks.jsonl written: {len(global_chunks)} chunks")
    return chunks_jsonl

# -------------------------------
# 7) Embeddings + FAISS for slides chunks
# -------------------------------
def build_slides_embeddings_and_faiss(chunks_jsonl: Path, emb_root=EMB_ROOT):
    # load chunks
    chunks = []
    with open(chunks_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                chunks.append(json.loads(line))

    texts = [c["text"] for c in chunks]
    print("Loaded slide chunks:", len(chunks))

    # SBERT
    print("Loading SBERT...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    print("Encoding...")
    embs = model.encode(texts, batch_size=16, show_progress_bar=True)
    embs = np.asarray(embs, dtype="float32")

    # save
    emb_path = emb_root / "slides_embeddings.npy"
    meta_path = emb_root / "slides_metadata.jsonl"
    faiss_path = emb_root / "slides_faiss_index_flatip.index"

    np.save(str(emb_path), embs)

    with open(meta_path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps({
                "chunk_id": c["chunk_id"],
                "pdf_stem": c["pdf_stem"],
                "slide_no": c["slide_no"],
                "source": c["source"],
            }, ensure_ascii=False) + "\n")

    # FAISS (cosine on normalized vectors)
    faiss.normalize_L2(embs)
    d = embs.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embs)
    faiss.write_index(index, str(faiss_path))

    print("✅ Saved:")
    print(" -", emb_path)
    print(" -", meta_path)
    print(" -", faiss_path)
    return emb_path, meta_path, faiss_path

# -------------------------------
# 8) (Optional) quick search demo
# -------------------------------
def demo_search(query: str, top_k: int = 5):
    emb_path = EMB_ROOT / "slides_embeddings.npy"
    meta_path = EMB_ROOT / "slides_metadata.jsonl"
    faiss_path = EMB_ROOT / "slides_faiss_index_flatip.index"
    chunks_path = OUT_ROOT / "slides_chunks.jsonl"

    if not (emb_path.exists() and meta_path.exists() and faiss_path.exists() and chunks_path.exists()):
        print("Missing slides index artifacts. Run embedding + faiss build first.")
        return

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    index = faiss.read_index(str(faiss_path))

    metadata = [json.loads(l) for l in meta_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    chunks = [json.loads(l) for l in chunks_path.read_text(encoding="utf-8").splitlines() if l.strip()]

    q = model.encode([query]).astype("float32")
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k)

    print("\n🔎 Query:", query)
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), start=1):
        if idx < 0:
            continue
        m = metadata[idx]
        c = chunks[idx]
        print(f"\n--- #{rank} | score={score:.3f} ---")
        print("pdf_stem:", m["pdf_stem"], "| slide_no:", m["slide_no"], "| chunk_id:", m["chunk_id"])
        print(c["text"][:450], "...")


# ==========================================================
# 9) RUN PIPELINE
# ==========================================================
slide_pdfs = sorted(SLIDES_DIR.glob("*.pdf"))
print("Slides PDFs found:", len(slide_pdfs))
if not slide_pdfs:
    raise FileNotFoundError(f"No PDFs found in {SLIDES_DIR}")

for pdf in slide_pdfs:
    process_slides_pdf(pdf)

chunks_path = build_slides_chunks()
build_slides_embeddings_and_faiss(chunks_path)

print("\n✅ NOTEBOOK 4 COMPLETED")
print("Outputs:")
print(" -", OUT_ROOT / "<pdf_stem>/slides_text_with_figures.txt")
print(" -", OUT_ROOT / "slides_chunks.jsonl")
print(" -", EMB_ROOT / "slides_faiss_index_flatip.index")

# Example search (optional)
demo_search("ternary relationship ER model", top_k=3)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Slides PDFs found: 8
✅ 1: slides processed = 33, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/1/figures
✅ 2: slides processed = 29, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/2/figures
✅ 3: slides processed = 28, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/3/figures
✅ 4: slides processed = 33, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/4/figures
✅ 5: slides processed = 41, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/5/figures
✅ 6: slides processed = 22, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/6/figures
✅ 7: slides processed = 28, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/7/figures
✅ 8: slides processed = 33, figures folder = /content/drive/MyDrive/RP/lecture_slides_extraction/8/figures
✅ slides_chunks.jsonl written: 55 chunks
Loaded slide chunks: 55
Loading SBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Saved:
 - /content/drive/MyDrive/RP/slides_embeddings/slides_embeddings.npy
 - /content/drive/MyDrive/RP/slides_embeddings/slides_metadata.jsonl
 - /content/drive/MyDrive/RP/slides_embeddings/slides_faiss_index_flatip.index

✅ NOTEBOOK 4 COMPLETED
Outputs:
 - /content/drive/MyDrive/RP/lecture_slides_extraction/<pdf_stem>/slides_text_with_figures.txt
 - /content/drive/MyDrive/RP/lecture_slides_extraction/slides_chunks.jsonl
 - /content/drive/MyDrive/RP/slides_embeddings/slides_faiss_index_flatip.index

🔎 Query: ternary relationship ER model

--- #1 | score=0.587 ---
pdf_stem: 1 | slide_no: 16 | chunk_id: 1__sc0003
the ternary relationship plus one or more of the binary relationships, if they represent different meanings and if all are needed --- SLIDE 16 --- ACTIVITY Draw an ER diagram for the scenario below. A Library is organized into several sections such as fiction, children and technology. Each section has a name and a number(unique) and its headed by a head librarian. Each bo